In [ ]:
!pip install -q transformers accelerate timm

# Multimodal models

### Types of Multimodal Models and Their Availability

| **Type**                          | **Input Modalities**         | **Output**    | **Typical Use Cases**              | **Popular Models**                          | **Free to Use?**        | **Model Size (Parameters)**                                                     |
| --------------------------------- | ---------------------------- | ------------- | ---------------------------------- | ------------------------------------------- | ----------------------- | ------------------------------------------------------------------------------- |
| **Image + Text → Text**           | Image + Text                 | Text          | VQA, captioning, image Q\&A        | BLIP ✅, BLIP-2 ✅, LLaVA ✅, MiniGPT-4 🟡     | ✅ (some fine-tuning 🟡) | BLIP: \~247M<br>BLIP-2: 2.7B–11B+<br>LLaVA: 13B<br>MiniGPT-4: 13B               |
| **Image → Text**                  | Image                        | Text          | Image captioning                   | BLIP ✅, GIT ✅, OFA ✅                        | ✅                       | BLIP: \~247M<br>GIT: 345M–1.2B<br>OFA: \~930M                                   |
| **Text → Image**                  | Text                         | Image         | Text-to-image generation           | DALL·E 🟡, Stable Diffusion ✅, Midjourney ❌ | ✅ (except Midjourney)   | DALL·E 2: Undisclosed (\~10B est.)<br>Stable Diffusion: \~890M<br>Midjourney: ❌ |
| **Text + Image → Image**          | Text + Image                 | Image         | Image editing, inpainting          | InstructPix2Pix ✅, ControlNet ✅             | ✅                       | InstructPix2Pix: \~900M<br>ControlNet: \~1.5B+                                  |
| **Image + Image → Text**          | Two images                   | Text          | Image comparison, change detection | GIT ✅, BLIP-2 ✅ (via prompting)             | ✅                       | GIT: 345M–1.2B<br>BLIP-2: 2.7B–11B+                                             |
| **Audio → Text**                  | Audio                        | Text          | Speech-to-text                     | Whisper ✅, Wav2Vec2 ✅                       | ✅                       | Whisper: 39M–1.5B<br>Wav2Vec2: \~317M                                           |
| **Text + Audio → Text**           | Audio + Text prompt          | Text          | Voice Q\&A                         | Whisper + LLM combo 🟡                      | ✅ (combo varies)        | Whisper: up to 1.5B<br>+ LLM: varies (7B–13B)                                   |
| **Video + Text → Text**           | Video + Question             | Text          | Video Q\&A, action recognition     | Flamingo 🟡, Video-BLIP ✅, GPT-4V ❌         | 🟡 (Flamingo limited)   | Flamingo: \~80B+<br>Video-BLIP: \~2.7B+<br>GPT-4V: ❌                            |
| **Text + Structure → Text**       | Text + Tables / JSON         | Text          | Data-to-text, table QA             | TAPAS ✅, T5+tables ✅, Gemini 🟡             | ✅ (Gemini limited)      | TAPAS: \~110M–440M<br>T5 (tables): \~220M–11B+<br>Gemini: ❌                     |
| **Image + Text → Classification** | Image + Text                 | Category      | Medical diagnosis, moderation      | CLIP ✅, MedCLIP ✅, ViLT ✅                   | ✅                       | CLIP: \~151M<br>MedCLIP: \~151M+<br>ViLT: \~86M                                 |
| **Multimodal Agents**             | Image + Text + Audio + Tools | Action / Text | AI assistants, tool-using agents   | ChatGPT-4o ❌, Gemini 1.5 🟡, MM-ReAct ✅     | 🟡/❌ (depends on stack) | GPT-4o: ❌<br>Gemini 1.5: ❌<br>MM-ReAct: \~13B+                                  |


# Load models

### `Salesforce/blip2-flan-t5-xl` — Model Overview

`blip2-flan-t5-xl` is a powerful **multimodal vision-language model** developed by Salesforce as part of the BLIP-2 framework (Bootstrapped Language-Image Pretraining). It combines a visual encoder (Vision Transformer) with a large instruction-tuned language model (FLAN-T5-XL) to enable **natural language understanding and generation grounded in images**.

| **Feature**               | **Description**                                                                 |
|---------------------------|---------------------------------------------------------------------------------|
| **Model Name**            | `Salesforce/blip2-flan-t5-xl`                                                  |
| **Architecture**          | BLIP-2 (Bootstrapping Language-Image Pretraining)                              |
| **Text Backbone**         | `FLAN-T5-XL` (3B parameters) — instruction-tuned T5                            |
| **Image Encoder**         | Vision Transformer (ViT-g / ViT-L variant depending on config)                 |
| **Input Modalities**      | Image + Text                                                                   |
| **Output Modality**       | Text (natural language response or generation)                                 |
| **Image Input Size**      | 224 × 224 px                                                                   |
| **Max Text Tokens (Input)** | 512 tokens                                                                   |
| **Pretrained Tasks**      | Image Captioning, Visual Question Answering, Instruction Following             |
| **Intended Use**          | Multimodal tasks: question answering, captioning, instruction-based generation |
| **Precision**             | Float16 (recommended) or Float32                                               |
| **Device Support**        | CPU / CUDA (with `device_map="auto"`)                                          |
| **Library**               | Hugging Face Transformers                                                      |
| **License**               | BSD 3-Clause (from Salesforce)                                                 |
| **Repository**            | [BLIP-2 on Hugging Face](https://huggingface.co/Salesforce/blip2-flan-t5-xl)   |


In [ ]:
from transformers import pipeline, Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import torch
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"

asr_pipe = pipeline("automatic-speech-recognition", model="openai/whisper-base")

# load BLIP-2 with FLAN-T5 (multimodal model: image + question)
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl") # 224 x 224 image, tokens: 512 text  BONUS: Salesforce/blip2-flan-t5-xxl
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1289 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie language_model.shared.weight to language_model.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

# Model Fuction Audio + Image -> Text

In [ ]:
def response_generation(audio_path, image_path):
    display(Audio(audio_path))

    # transcribe the audio using Whisper
    transcription = asr_pipe(audio_path)["text"]
    print(f"\nTranscription: {transcription}")

    # load and display the image
    image = Image.open(image_path).convert("RGB")
    display(image)

    # create prompt based on user transcription
    prompt = f"Question: {transcription}"

    # prepare inputs
    inputs = processor(image, prompt, return_tensors="pt").to(device)

    #generate answer
    outputs = model.generate(**inputs, max_new_tokens=200)
    answer = processor.decode(outputs[0], skip_special_tokens=True)

    # print final result
    print("\nAnswer of the multimodal agent:")
    print(f"User question from audio: {transcription}")
    print(f"Model response based on image and question: {answer}")

# Test models

In [ ]:
audio_path = "/kaggle/input/interactive-multimodal-agent-dataset/color_car.mp3"

image_path = "/kaggle/input/interactive-multimodal-agent-dataset/car.jpg"

response_generation(audio_path, image_path)

ValueError: rate must be specified when data is a numpy array or list of audio samples.

In [ ]:
response_generation("/kaggle/input/interactive-multimodal-agent-dataset/emotion.mp3", "/kaggle/input/interactive-multimodal-agent-dataset/Happy.jpg")